# Pra-pemrosesan teks - Klasifikasi Teks Berita Kesehatan Berbahasa Indonesia
### Model: Convolutional Long Short-Term Memory (C-LSTM)
**Dataset:** Berita kesehatan berlabel `diabetes` dan `tbc`

---

In [36]:
import pandas as pd
df = pd.read_csv("data/dataset.csv")

In [37]:
# ── Cek kondisi sebelum cleaning ──────────────────────────────
print("=" * 50)
print("  KONDISI SEBELUM CLEANING")
print("=" * 50)
print(f"  Total baris          : {len(df):,}")
print()
print("  Missing values       :")
print(f"    content (NaN)      : {df['content'].isna().sum()}")
print(f"    content (kosong)   : {(df['content'] == '').sum()}")
print(f"    title (NaN)        : {df['title'].isna().sum()}")
print()
print("  Duplikat             :")
print(f"    Duplikat judul     : {df['title'].duplicated().sum():,}")
print(f"    Duplikat konten    : {df['content'].duplicated().sum():,}")
print()
print("  Distribusi label     :")
print(df['keyword'].value_counts().to_string())

  KONDISI SEBELUM CLEANING
  Total baris          : 2,519

  Missing values       :
    content (NaN)      : 22
    content (kosong)   : 0
    title (NaN)        : 0

  Duplikat             :
    Duplikat judul     : 261
    Duplikat konten    : 283

  Distribusi label     :
keyword
diabetes    1355
tbc         1164


## 1. Menghapus baris content Nan

In [38]:
# ── Step 1: Hapus baris dengan content NaN ────────────────────
before = len(df)
df = df.dropna(subset=['content'])
dropped_nan = before - len(df)
print(f"  Baris dihapus (NaN)   : {dropped_nan}")

# ── Step 2: Hapus baris dengan content string kosong ──────────
before = len(df)
df = df[df['content'].str.strip() != '']
dropped_empty = before - len(df)
print(f"  Baris dihapus (kosong): {dropped_empty}")

print(f"  Sisa baris            : {len(df):,}")

  Baris dihapus (NaN)   : 22
  Baris dihapus (kosong): 0
  Sisa baris            : 2,497


## 2. Menghapus duplikat Content & Title

In [39]:
# ── Step 3: Hapus duplikat berdasarkan konten ─────────────────
# PENTING: lakukan SEBELUM split train/val/test
# agar tidak terjadi data leakage antar set

before = len(df)
df = df.drop_duplicates(subset=['content'], keep='first')
dropped_dup = before - len(df)
print(f"  Duplikat dihapus (content)     : {dropped_dup:,}")

# hapus  duplikat judul
before_title = len(df)
df = df.drop_duplicates(subset=['title'], keep='first')
dropped_dup_title = before_title - len(df)
print(f"  Duplikat dihapus (title)       : {dropped_dup_title:,}")

print(f"  Sisa baris            : {len(df):,}")

  Duplikat dihapus (content)     : 262
  Duplikat dihapus (title)       : 1
  Sisa baris            : 2,234


## 3. Ringkasan Setelah Cleaning

In [40]:
# ── Step 4: Reset index ───────────────────────────────────────
df = df.reset_index(drop=True)

print("=" * 50)
print("  KONDISI SETELAH CLEANING")
print("=" * 50)
print(f"  Total baris          : {len(df):,}")
print()
print("  Missing values       :")
print(f"    content (NaN)      : {df['content'].isna().sum()}")
print(f"    content (kosong)   : {(df['content'] == '').sum()}")
print()
print("  Duplikat             :")
print(f"    Duplikat konten    : {df['content'].duplicated().sum()}")
print()
print("  Distribusi label     :")
print(df['keyword'].value_counts().to_string())
print()
print("  Ringkasan perubahan  :")
print(f"    Dihapus (NaN)      : {dropped_nan}")
print(f"    Dihapus (kosong)   : {dropped_empty}")
total_dup = dropped_dup + dropped_dup_title
print(f"    Dihapus (duplikat) : {total_dup}")
print(f"    Total dihapus      : {dropped_nan + dropped_empty + dropped_dup}")

  KONDISI SETELAH CLEANING
  Total baris          : 2,234

  Missing values       :
    content (NaN)      : 0
    content (kosong)   : 0

  Duplikat             :
    Duplikat konten    : 0

  Distribusi label     :
keyword
diabetes    1184
tbc         1050

  Ringkasan perubahan  :
    Dihapus (NaN)      : 22
    Dihapus (kosong)   : 0
    Dihapus (duplikat) : 263
    Total dihapus      : 284


## 4. Menghapus Kolom Tidak Penting

In [41]:
df = df.drop(columns=['Unnamed: 0', 'Unnamed: 0.1', 'Unnamed: 0.2'])
df

,title,link,keyword,content
0,Andre Rosiade Bantu Istri Pemulung Pengidap Di...,https://news.detik.com/berita/d-8452634/andre-...,diabetes,Wakil Ketua Komisi VI DPR RI dari Fraksi Gerin...
1,Pasien Diabetes Tak Lagi Perlu Minum Obat Seum...,https://health.detik.com/berita-detikhealth/d-...,diabetes,Ketua Umum Perkumpulan Endokrinologi Indonesia...
2,18 April Hari Diabetes Nasional dan Kenali Car...,https://www.detik.com/bali/berita/d-8450035/18...,diabetes,"Setiap tahun, tanggal 18 April selalu dipering..."
3,Terungkap Pemicu Terbesar Gagal Ginjal Makin M...,https://health.detik.com/berita-detikhealth/d-...,diabetes,Kementerian Kesehatan RI mengungkap tren mengk...
4,Jebakan 'Hidden Sugar' di Minuman Manis yang B...,https://health.detik.com/berita-detikhealth/d-...,diabetes,Kabar baik bagi kesehatan publik di Indonesia....
...,...,...,...,...
2229,"Dolan ke Balai Kota Surabaya,Â Ini yang Dibaha...",https://www.detik.com/jatim/berita/d-6023519/d...,tbc,Gubernur Jatim Khofifah Indar Parawansa untuk ...
2230,Pemkot Tangerang Raih 48 Penghargaan Sepanjang...,https://news.detik.com/berita/d-6478020/pemkot...,tbc,"Sepanjang tahun 2022, Pemerintah Kota (Pemkot)..."
2231,Kata Dokter Paru soal 619 Anak di Bantul Kena ...,https://health.detik.com/berita-detikhealth/d-...,tbc,Dinas Kesehatan Kabupaten Bantul melaporkan ad...
2232,Jepang yang Bakal Wajibkan WNI Tes TBC Mulai 2...,https://health.detik.com/berita-detikhealth/d-...,tbc,Kementerian Kesehatan RI (Kemenkes) menanggapi...


## 5. Cek Empty DataFrame

In [42]:
df['word_count'] = df['content'].str.split().apply(len)
print(df[df['word_count'] == 0])

Empty DataFrame
Columns: [title, link, keyword, content, word_count]
Index: []


## 6. Casefolding

In [43]:
X = df['content']     # fitur
y = df['keyword']     # label

df['content_case'] = df['content'].str.lower()
df['content_case']

0       wakil ketua komisi vi dpr ri dari fraksi gerin...
1       ketua umum perkumpulan endokrinologi indonesia...
2       setiap tahun, tanggal 18 april selalu dipering...
3       kementerian kesehatan ri mengungkap tren mengk...
4       kabar baik bagi kesehatan publik di indonesia....
                              ...                        
2229    gubernur jatim khofifah indar parawansa untuk ...
2230    sepanjang tahun 2022, pemerintah kota (pemkot)...
2231    dinas kesehatan kabupaten bantul melaporkan ad...
2232    kementerian kesehatan ri (kemenkes) menanggapi...
2233    misteri kematian pria di lubuklinggau, sumater...
Name: content_case, Length: 2234, dtype: object

## 7. Special character removal

In [44]:
import re

def clean_text(text):
    text = str(text)
    
    # hapus URL
    text = re.sub(r'http\S+|www\S+', '', text)
    
    # hapus mention & hashtag
    text = re.sub(r'@\w+|#\w+', '', text)
    
    # hapus angka
    text = re.sub(r'\d+', '', text)
    
    # hapus tanda baca & karakter khusus
    text = re.sub(r'[^\w\s]', '', text)
    
    # hapus spasi berlebih
    text = re.sub(r'\s+', ' ', text).strip()
    
    return text

# terapkan ke data
df['content_clean'] = df['content_case'].apply(clean_text)
df[['content_case', 'content_clean']].head(10)

,content_case,content_clean
0,wakil ketua komisi vi dpr ri dari fraksi gerin...,wakil ketua komisi vi dpr ri dari fraksi gerin...
1,ketua umum perkumpulan endokrinologi indonesia...,ketua umum perkumpulan endokrinologi indonesia...
2,"setiap tahun, tanggal 18 april selalu dipering...",setiap tahun tanggal april selalu diperingati ...
3,kementerian kesehatan ri mengungkap tren mengk...,kementerian kesehatan ri mengungkap tren mengk...
4,kabar baik bagi kesehatan publik di indonesia....,kabar baik bagi kesehatan publik di indonesia ...
5,badan pengawas obat dan makanan (bpom) menyetu...,badan pengawas obat dan makanan bpom menyetuju...
6,kementerian kesehatan (kemenkes) bersama jajar...,kementerian kesehatan kemenkes bersama jajaran...
7,tak sedikit pasien diabetes yang cemas saat me...,tak sedikit pasien diabetes yang cemas saat me...
8,diabetes dikenal sebagai penyakit yang berbaha...,diabetes dikenal sebagai penyakit yang berbaha...
9,ingin ngemil tapi takut gula darah naik? kekha...,ingin ngemil tapi takut gula darah naik kekhaw...


## Tokenisasi

In [49]:
df['tokens'] = df['content_clean'].apply(lambda x: x.split())
df[['content_clean', 'tokens']].head(10)

,content_clean,tokens
0,wakil ketua komisi vi dpr ri dari fraksi gerin...,"[wakil, ketua, komisi, vi, dpr, ri, dari, frak..."
1,ketua umum perkumpulan endokrinologi indonesia...,"[ketua, umum, perkumpulan, endokrinologi, indo..."
2,setiap tahun tanggal april selalu diperingati ...,"[setiap, tahun, tanggal, april, selalu, diperi..."
3,kementerian kesehatan ri mengungkap tren mengk...,"[kementerian, kesehatan, ri, mengungkap, tren,..."
4,kabar baik bagi kesehatan publik di indonesia ...,"[kabar, baik, bagi, kesehatan, publik, di, ind..."
5,badan pengawas obat dan makanan bpom menyetuju...,"[badan, pengawas, obat, dan, makanan, bpom, me..."
6,kementerian kesehatan kemenkes bersama jajaran...,"[kementerian, kesehatan, kemenkes, bersama, ja..."
7,tak sedikit pasien diabetes yang cemas saat me...,"[tak, sedikit, pasien, diabetes, yang, cemas, ..."
8,diabetes dikenal sebagai penyakit yang berbaha...,"[diabetes, dikenal, sebagai, penyakit, yang, b..."
9,ingin ngemil tapi takut gula darah naik kekhaw...,"[ingin, ngemil, tapi, takut, gula, darah, naik..."


## 8. Stopword Removal

In [46]:
# %pip install Sastrawi

In [52]:
from Sastrawi.StopWordRemover.StopWordRemoverFactory import StopWordRemoverFactory

factory = StopWordRemoverFactory()
stopwords = set(factory.get_stop_words())

custom_stopwords = {
    'baca', 'juga', 'detikcom', 'kompas', 'tribun',
    'co', 'id', 'www', 'https', 'html', 'kumparan'
}

stopwords = stopwords.union(custom_stopwords)

def remove_stopwords(tokens):
    return [word for word in tokens if word not in stopwords]

df['tokens_stop'] = df['tokens'].apply(remove_stopwords)
df[['tokens', 'tokens_stop']].to_excel('hasil_stopword.xlsx', index=False)
df[['tokens','tokens_stop']].head(5)

,tokens,tokens_stop
0,"[wakil, ketua, komisi, vi, dpr, ri, dari, frak...","[wakil, ketua, komisi, vi, dpr, ri, fraksi, ge..."
1,"[ketua, umum, perkumpulan, endokrinologi, indo...","[ketua, umum, perkumpulan, endokrinologi, indo..."
2,"[setiap, tahun, tanggal, april, selalu, diperi...","[tahun, tanggal, april, selalu, diperingati, h..."
3,"[kementerian, kesehatan, ri, mengungkap, tren,...","[kementerian, kesehatan, ri, mengungkap, tren,..."
4,"[kabar, baik, bagi, kesehatan, publik, di, ind...","[kabar, baik, kesehatan, publik, indonesia, ba..."


## 9. Stemming

In [53]:
from Sastrawi.Stemmer.StemmerFactory import StemmerFactory

factory_stem = StemmerFactory()
stemmer = factory_stem.create_stemmer()

def stemming(tokens):
    return [stemmer.stem(word) for word in tokens]

df['tokens_stem'] = df['tokens_stop'].apply(stemming)
df[['tokens_stop','tokens_stem']].to_excel('hasil_stemming.xlsx', index=False)
df[['tokens_stop','tokens_stem']].head(5)

,tokens_stop,tokens_stem
0,"[wakil, ketua, komisi, vi, dpr, ri, fraksi, ge...","[wakil, ketua, komisi, vi, dpr, ri, fraksi, ge..."
1,"[ketua, umum, perkumpulan, endokrinologi, indo...","[ketua, umum, kumpul, endokrinologi, indonesia..."
2,"[tahun, tanggal, april, selalu, diperingati, h...","[tahun, tanggal, april, selalu, ingat, hari, d..."
3,"[kementerian, kesehatan, ri, mengungkap, tren,...","[menteri, sehat, ri, ungkap, tren, khawatir, k..."
4,"[kabar, baik, kesehatan, publik, indonesia, ba...","[kabar, baik, sehat, publik, indonesia, badan,..."


In [54]:
df['content_final'] = df['tokens_stem'].apply(
    lambda x: ' '.join(x) if isinstance(x, list) else ''
)

df.to_csv('hasil_preprocessing.csv', index=False, encoding='utf-8-sig')

df[['tokens_stem', 'content_final']].head(5)

,tokens_stem,content_final
0,"[wakil, ketua, komisi, vi, dpr, ri, fraksi, ge...",wakil ketua komisi vi dpr ri fraksi gerindra a...
1,"[ketua, umum, kumpul, endokrinologi, indonesia...",ketua umum kumpul endokrinologi indonesia ken ...
2,"[tahun, tanggal, april, selalu, ingat, hari, d...",tahun tanggal april selalu ingat hari diabetes...
3,"[menteri, sehat, ri, ungkap, tren, khawatir, k...",menteri sehat ri ungkap tren khawatir kait tin...
4,"[kabar, baik, sehat, publik, indonesia, badan,...",kabar baik sehat publik indonesia badan awas o...
